In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print(
        "There might be a problem with your API key? Please visit the troubleshooting notebook!"
    )

MODEL = "gpt-5-nano"
openai = OpenAI()


In [ ]:
links = fetch_website_links("https://www.budgetbytes.com/category/recipes/one-pot/")
links


In [ ]:
link_system_prompt = """
You are given a list of links from a cooking website. Pick the 3 recipes easiest for a COMPLETE BEGINNER, ranked by (in order):
1. Short total time (ideally <30 min)
2. Only basic equipment (pan, pot, oven, knife, stovetop — no stand mixer, food processor, thermometer, pressure cooker, sous vide, etc.)
3. Few, common ingredients
4. Simple techniques (no deboning, folding, tempering, proofing, laminating, multi-stage cooking)

Ignore non-recipe links (category, blog, about, shop, login, legal, email).

Respond in JSON:
{"recipes":[
  {"rank":1,"title":"One-Pan Garlic Butter Pasta","url":"https://...","reason":"15 min, one pan, 5 ingredients"},
  {"rank":2,"title":"Scrambled Eggs on Toast","url":"https://...","reason":"10 min, basic pan, no knife skills"},
  {"rank":3,"title":"Microwave Mug Oatmeal","url":"https://...","reason":"3 min, microwave, 3 ingredients"}
]}
"""


In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the cooking website {url} -
Please decide the 3 EASIEST recipes for a complete beginner to make, prioritising fast cook time and minimal kitchen equipment. Respond with the full https URL in JSON format.
Do not include category pages, blog posts that aren't recipes, login/register pages, Terms of Service, Privacy, or email links.
Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
print(get_links_user_prompt("https://www.budgetbytes.com/category/recipes/one-pot/"))


In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['recipes'])} relevant links")
    return links


In [ ]:
select_relevant_links("https://www.budgetbytes.com/category/recipes/one-pot/")


## Making the brochure with the cooking recipes.


In [ ]:
# format the user prompt first.
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Top 3 Recipes:\n"
    for link in relevant_links["recipes"]:
        result += f"\n\n### Recipe name: {link['title']}\n"
        result += f"\n\n### Reason for selection: {link['reason']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
print(
    fetch_page_and_all_relevant_links(
        "https://www.budgetbytes.com/category/recipes/one-pot/"
    )
)


In [ ]:
cookbook_system_prompt = """
You produce a short, friendly "Beginner's Cookbook" in markdown from 3 pre-selected easy recipes (raw scraped content supplied).

For EACH recipe, output a section with:
- A one-line confidence-boosting intro
- **Total time** and **servings**
- **Equipment** (basics only)
- **Ingredients** as bullets, with common substitutions in brackets
- **Steps**: numbered, plain encouraging language; define any cooking term on first use (e.g. "sauté = cook in a little oil over medium heat until soft")
- **Beginner tip**: the single most common mistake to avoid

End with "What to cook first" — pick ONE recipe as the best starting point, one-sentence reason.

Be warm, concise, never condescending. Do NOT invent ingredients or steps beyond the scraped content — only rephrase and clarify.
"""


In [ ]:
def get_cookbook_user_prompt(url):
    user_prompt = f"""
Here are 3 beginner-friendly recipes scraped from {url}. Please turn them into a short Beginner's Cookbook in markdown, following the format in your system prompt.
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[
        :5_000
    ]  # Truncate if more than 5,000 characters (can use 5000, 5_000 is for readability)
    return user_prompt

In [ ]:
get_cookbook_user_prompt("https://www.budgetbytes.com/category/recipes/one-pot/")


### Stream the cookbook


In [ ]:
def stream_cookbook(url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": cookbook_system_prompt},
            {"role": "user", "content": get_cookbook_user_prompt(url)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
stream_cookbook("https://www.budgetbytes.com/category/recipes/one-pot/")